### 연습
- ratings_train.txt 파일을 로드
- 결측치 제거
- document 컬럼의 문자 정규화(특수문자 제거, 2칸 이상의 공백 제거, 좌우 공백 제거)
- 중복 document 제거 , 글자의 수가 1개 이하인 행은 제거
- DataFrame에서 sample(n = 10000, random_state=42)로 임의의 데이터를 추출하여 저장 (head() -> 상위 데이터 | tail() -> 하위 데이터 | sample() -> 무작위 데이터)
- train, test 셋으로 8:2 로 데이터분할
- sbert 모델은 'BM-K/KoSimCSE-roberta-multitask'을 이용
- Dataset을 정의 (Trainer 이용하지 않고 Dataset과 DataLoader 사용)
    - 입력받은 document와 label를 document는 SBERT 모델을 이용하여 인코딩
    - label 데이터를 tensor형태로 변환
    - __len__ 함수는 라벨의 길이를 되돌려준다
    - __getitem__ 함수는 인코딩된 데이터[idx], label[idx]를 되돌려준다
- Dataset를 train, test를 이용해서 Dataset을 생성
- DataLoarder를 이용하여 배치의 사이즈는 128 shuffle은 True 구성한다.

In [1]:
import pandas as pd
import re
import torch
from sentence_transformers import SentenceTransformer, util
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('../data/ratings_train.txt', sep='\t')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        150000 non-null  int64 
 1   document  149995 non-null  object
 2   label     150000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.4+ MB


In [3]:
df.dropna(inplace=True)

In [4]:
def normalize(text):
    text = re.sub(r'[^가-힣0-9a-zA-Z\s\.]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [5]:
df['document'] = df['document'].map(normalize)

In [6]:
df = df.loc[df['document'].str.len() > 1]

In [7]:
df = df.drop_duplicates('document')
df

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화 스파이더맨에서 늙어보이기만 했던 커스틴 ...,1
...,...,...,...
149995,6222902,인간이 문제지.. 소는 뭔죄인가..,0
149996,8549745,평점이 너무 낮아서...,1
149997,9311800,이게 뭐요 한국인은 거들먹거리고 필리핀 혼혈은 착하다,0
149998,2376369,청춘 영화의 최고봉.방황과 우울했던 날들의 자화상,1


In [8]:
df2 = df.sample(n = 10000, random_state=42)

In [9]:
train_df, test_df = train_test_split(
    df2, test_size=0.2, random_state=42, stratify=df2['label']
)

In [10]:
model_name = 'BM-K/KoSimCSE-roberta-multitask'
sbert = SentenceTransformer(model_name)

No sentence-transformers model found with name BM-K/KoSimCSE-roberta-multitask. Creating a new one with mean pooling.


In [11]:
class SBERTDataset(Dataset):
    def __init__(self, document, labels):
        self.embeddings = sbert.encode(document, convert_to_tensor=True)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]

In [12]:
train_dataset = SBERTDataset(
    train_df['document'].values,  # .values 추가
    train_df['label'].values
)

test_dataset = SBERTDataset(
    test_df['document'].values,
    test_df['label'].values
)

In [15]:
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

In [16]:
train_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=True
)